In [ ]:
import os
import time
import math
import pandas as pd
import numpy as np
import pylab as plt
import seaborn as sns

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black
jupyter_black.load()

In [ ]:
plt.style.use("../src/mpl_style.txt")

In [ ]:
RESULTS_PATH = "../results/baseline_2025-06-26/research-article_aimrd_f"

In [ ]:
selection = ["delve", "crucial", "potential", "these", "significant", "important"]

In [ ]:
start_date = "11-2020"  # this is the first date being used for projection
end_date = "6-2025"

In [ ]:
months = np.arange(1, 13)
years = np.arange(2000, 2025)
x_months = np.arange(2020 + (10 / 12), 2025 + (5 / 12), 1 / 12)

In [ ]:
frequency_dfs = utils.load_freqs(
    RESULTS_PATH, selection, start_date, end_date, x_months
)

In [ ]:
frequency_dfs["potential"]

In [ ]:
freq_dfs_agg = {}

for word, df in frequency_dfs.items():
    df = df.copy()
    df["time"] = list(map(math.floor, df["time"]))
    df = df.groupby(["time", "section"]).mean().reset_index()
    df["section"] = pd.Categorical(
        df["section"],
        categories=[
            "abstract",
            "introduction",
            "methods",
            "results",
            "discussion",
            "full",
        ],
        ordered=True,
    )

    freq_dfs_agg[word] = df

In [ ]:
freq_dfs_agg["potential"]["section"].cat.categories

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(7, 4.5), layout="constrained")

for i, word in enumerate(selection):
    ax = axs.flat[i]
    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(
        data=frequency_dfs[word], x="time", y="frequency", hue="section", ax=ax
    )
    sns.lineplot(
        data=frequency_dfs[word],
        x="time",
        y="projection",
        hue="section",
        ax=ax,
        alpha=0.5,
        linestyle="-.",
        legend=False,
    )
    ax.set_title(word)
    ax.set_xlabel(None)
    if i < 3:
        ax.set_xticks([2021, 2022, 2023, 2024, 2025])
        ax.set_xticklabels([])
    else:
        ax.set_xticks([2021, 2022, 2023, 2024, 2025])
        ax.set_xticklabels([2021, 2022, 2023, 2024, 2025], rotation=60)
    if i not in [0, 3]:
        ax.set_ylabel(None)
    if not i == 0:
        ax.get_legend().set_visible(False)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=6, figsize=(10, 4), layout="constrained")

for i, word in enumerate(selection):

    # plot diffs
    ax = axs[0, i]
    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(data=frequency_dfs[word], x="time", y="diff", hue="section", ax=ax)

    # plot ratios
    ax = axs[1, i]
    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(data=frequency_dfs[word], x="time", y="ratio", hue="section", ax=ax)

    for j in range(2):
        ax = axs[j, i]
        ax.set_xlabel(None)
        if j == 0:
            ax.set_title(word)
        if j == 1:
            ax.set_xticks([2021, 2022, 2023, 2024, 2025])
            ax.set_xticklabels([2021, 2022, 2023, 2024, 2025], rotation=60)
        else:
            ax.set_xticks([2021, 2022, 2023, 2024, 2025])
            ax.set_xticklabels([])
        if not i == 0:
            ax.set_ylabel(None)
        if not (i == 0 and j == 0):
            ax.get_legend().set_visible(False)

is there a way to standardize between sections? bc the longer a section is, the more likely it is that a word occurs? or give mean word counts per sections?

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=6, figsize=(8, 4), layout="constrained")

for i, word in enumerate(selection):

    # plot diffs
    ax = axs[0, i]
    sns.pointplot(data=freq_dfs_agg[word], x="section", y="diff", hue="time", ax=ax)

    # plot ratios
    ax = axs[1, i]
    sns.pointplot(data=freq_dfs_agg[word], x="section", y="ratio", hue="time", ax=ax)

    for j in range(2):
        ax = axs[j, i]
        ax.set_xlabel(None)
        if j == 0:
            ax.set_title(word)
        if j == 1:
            ax.set_xticklabels(
                freq_dfs_agg["potential"]["section"].cat.categories, rotation=60
            )
        else:
            ax.set_xticklabels([])
        if not i == 0:
            ax.set_ylabel(None)
        if not (i == 0 and j == 0):
            ax.get_legend().set_visible(False)